<a href="https://colab.research.google.com/github/Data-Creater-Atlas/Data-Atlas/blob/Seongjong/crossattention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import re
import random
import time
from pathlib import Path
from typing import List, Tuple, Optional

import numpy as np
import cv2
import tifffile as tiff
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from tqdm.auto import tqdm

from google.colab import drive

# 1. 드라이브 마운트
if not os.path.exists("/content/drive"):
    drive.mount("/content/drive")

# ------------------------------------------------------------
# CFG: 설정 (경로, 파라미터)
# ------------------------------------------------------------
CFG = {
    # ☠  사용자 경로 확인
    "DATA_ROOT": "/content/drive/MyDrive/Data Camp/53.대기오염 배출원 공간 분포 데이터/3.개방데이터/1.데이터",

    "TRAIN_IMG_DIR": "Training/01.원천데이터/TS_SN10_SN10",
    "TRAIN_MASK_DIR": "Training/02.라벨링데이터/TL_SN10",
    "VALID_IMG_DIR": "Validation/01.원천데이터/VS_SN10_SN10",
    "VALID_MASK_DIR": "Validation/02.라벨링데이터/VL_SN10",

    "TRAIN_NO2_DIR": "Training/01.원천데이터/TS_SN10_GEMS",
    "TRAIN_AIR_DIR": "Training/01.원천데이터/TS_SN10_AIR_Pollution",
    "VALID_NO2_DIR": "Validation/01.원천데이터/VS_SN10_GEMS",
    "VALID_AIR_DIR": "Validation/01.원천데이터/VS_SN10_AIR_Pollution",

    "IMAGE_EXT": ".tif",
    "AUX_EXTS": [".tif", ".tiff", ".npy"],

    "CLIP_MAX": 10000.0,
    "IMG_SIZE": 256,

    "BATCH_SIZE": 8,    # OOM 발생 시 8로 줄이세요
    "NUM_WORKERS": 2,
    "EPOCHS": 150,
    "LR": 1e-4,
    "WD": 1e-4,
    "SEED": 42,

    # 전처리 통계 (Log Scale 사용)
    "AUTO_COMPUTE_AUX_STATS": True,
    "NO2_MEAN": 0.0, "NO2_STD": 1.0,
    "AIR_MEAN": 0.0, "AIR_STD": 1.0,

    "ES_PATIENCE": 15,
    "ES_MIN_DELTA": 1e-4,
    "ES_MONITOR": "miou",
    "ES_RESTORE_BEST": True,

    "CHECKPOINT": "sn10_5ch_transunet_aspp_gated_crossattn.pt",
}

# ============================================================
# 1. 유틸리티 및 전처리
# ============================================================
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def read_tif_any(path: Path):
    path = Path(path)
    pstr = str(path)
    try: return tiff.imread(pstr)
    except: pass
    try:
        with Image.open(pstr) as im: return np.array(im)
    except: pass
    arr = cv2.imread(pstr, cv2.IMREAD_UNCHANGED)
    if arr is None: raise FileNotFoundError(f"Failed to read TIF: {pstr}")
    return arr

def load_image_tif(path: Path, clip_max: float = 10000.0) -> np.ndarray:
    arr = read_tif_any(path)
    if arr.ndim == 2: arr = np.stack([arr] * 3, axis=-1)
    elif arr.ndim == 3 and arr.shape[-1] > 3: arr = arr[..., :3]
    arr = arr.astype(np.float32)
    arr = np.clip(arr, 0, clip_max) / clip_max
    arr = np.nan_to_num(arr, nan=0.0)
    return arr

def load_mask_tif_binary(path: Path, positive_value: int = 10) -> np.ndarray:
    m = read_tif_any(path)
    if m.ndim == 3: m = m[..., 0]
    return (m == positive_value).astype(np.float32)

# ★ 로그 스케일 전처리
def prepare_aux_raw(arr: np.ndarray) -> np.ndarray:
    if arr.ndim == 3: arr = arr.mean(axis=-1)
    arr = arr.astype(np.float32)
    vmax = float(np.nanmax(arr))
    if vmax > 1e6: arr = arr / 1e16
    arr = np.maximum(arr, 0.0)
    arr = np.log1p(arr) # Log Scale
    return arr

def collect_aux_stats(aux_dir: Path) -> Tuple[float, float]:
    aux_dir = Path(aux_dir)
    paths = sorted(aux_dir.glob("*.tif"))
    if not paths: return 0.0, 1.0
    vals = []
    for p in tqdm(paths, desc=f"[Stats] {aux_dir.name}"):
        try:
            arr = read_tif_any(p)
            arr = prepare_aux_raw(arr)
            vals.append(arr.flatten())
        except: continue
    vals = np.concatenate(vals)
    return float(vals.mean()), float(vals.std())

def load_aux_map(path: Optional[Path], target_hw: Tuple[int, int], mean: float, std: float) -> np.ndarray:
    H, W = target_hw
    if path is None or (not path.exists()): return np.zeros((H, W), dtype=np.float32)
    arr = read_tif_any(path)
    arr = prepare_aux_raw(arr)
    if arr.shape[:2] != (H, W):
        arr = cv2.resize(arr, (W, H), interpolation=cv2.INTER_LINEAR)
    if std == 0.0: std = 1.0
    arr = (arr - mean) / std
    return arr.astype(np.float32)

# ============================================================
# 2. 데이터셋 (5채널 구성)
# ============================================================
def extract_sn10_key(path: Path) -> Optional[str]:
    m = re.search(r"(\d{5}_\d{6})", path.stem)
    return m.group(1) if m else None

def list_pairs_with_aux(img_dir, mask_dir, no2_dir, air_dir, ext, aux_exts):
    img_paths = sorted(img_dir.glob(f"*{ext}"))
    no2_index = {extract_sn10_key(f): f for f in no2_dir.glob("*.tif") if extract_sn10_key(f)}
    air_index = {extract_sn10_key(f): f for f in air_dir.glob("*.tif") if extract_sn10_key(f)}

    items = []
    for img_p in img_paths:
        key = extract_sn10_key(img_p)
        if not key: continue
        mask_p = mask_dir / f"{img_p.stem}{ext}"
        if not mask_p.exists(): continue
        items.append((img_p, mask_p, no2_index.get(key), air_index.get(key)))
    return items

class SN10MultiModal5chDataset(Dataset):
    def __init__(self, items, img_size=256, is_train=True):
        self.items = items
        self.img_size = img_size
        self.is_train = is_train

    def __len__(self): return len(self.items)

    def __getitem__(self, idx):
        img_p, mask_p, no2_p, air_p = self.items[idx]

        img = load_image_tif(img_p, clip_max=CFG["CLIP_MAX"])
        mask = load_mask_tif_binary(mask_p)
        H0, W0 = mask.shape[:2]

        no2 = load_aux_map(no2_p, (H0, W0), CFG["NO2_MEAN"], CFG["NO2_STD"])
        air = load_aux_map(air_p, (H0, W0), CFG["AIR_MEAN"], CFG["AIR_STD"])

        H, W = self.img_size, self.img_size
        img = cv2.resize(img, (W, H), interpolation=cv2.INTER_LINEAR)
        no2 = cv2.resize(no2, (W, H), interpolation=cv2.INTER_LINEAR)
        air = cv2.resize(air, (W, H), interpolation=cv2.INTER_LINEAR)
        mask = cv2.resize(mask, (W, H), interpolation=cv2.INTER_NEAREST)

        if self.is_train:
            if np.random.rand() < 0.5:
                img, no2, air, mask = [np.flip(x, axis=1).copy() for x in [img, no2, air, mask]]
            if np.random.rand() < 0.5:
                img, no2, air, mask = [np.flip(x, axis=0).copy() for x in [img, no2, air, mask]]
            if np.random.rand() < 0.5:
                k = np.random.randint(1, 4)
                img, no2, air, mask = [np.rot90(x, k=k, axes=(0, 1)).copy() for x in [img, no2, air, mask]]

        img_t = torch.from_numpy(img.transpose(2, 0, 1)).float()
        no2_t = torch.from_numpy(no2).unsqueeze(0).float()
        air_t = torch.from_numpy(air).unsqueeze(0).float()
        x = torch.cat([img_t, no2_t, air_t], dim=0)
        y = torch.from_numpy(mask).unsqueeze(0).float()
        return x, y

def make_loaders_5ch():
    root = Path(CFG["DATA_ROOT"])
    tr_items = list_pairs_with_aux(root/CFG["TRAIN_IMG_DIR"], root/CFG["TRAIN_MASK_DIR"], root/CFG["TRAIN_NO2_DIR"], root/CFG["TRAIN_AIR_DIR"], CFG["IMAGE_EXT"], CFG["AUX_EXTS"])
    va_items = list_pairs_with_aux(root/CFG["VALID_IMG_DIR"], root/CFG["VALID_MASK_DIR"], root/CFG["VALID_NO2_DIR"], root/CFG["VALID_AIR_DIR"], CFG["IMAGE_EXT"], CFG["AUX_EXTS"])

    train_ds = SN10MultiModal5chDataset(tr_items, img_size=CFG["IMG_SIZE"], is_train=True)
    valid_ds = SN10MultiModal5chDataset(va_items, img_size=CFG["IMG_SIZE"], is_train=False)

    train_loader = DataLoader(train_ds, batch_size=CFG["BATCH_SIZE"], shuffle=True, num_workers=CFG["NUM_WORKERS"], pin_memory=True, drop_last=True)
    valid_loader = DataLoader(valid_ds, batch_size=CFG["BATCH_SIZE"], shuffle=False, num_workers=CFG["NUM_WORKERS"], pin_memory=True, drop_last=False)
    return train_loader, valid_loader

# ============================================================
# 3. 모델: TransUNet + ASPP + Gated Fusion + Cross Attention (SEBlock 제거됨)
# ============================================================
class ConvBlock(nn.Module):
    """ Standard Conv Block (No SE) """
    def __init__(self, in_c, out_c):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c), nn.ReLU(inplace=True)
        )
    def forward(self, x): return self.conv(x)

class ASPP(nn.Module):
    def __init__(self, in_ch, out_ch=256, rates=(1, 6, 12, 18)):
        super().__init__()
        self.blocks = nn.ModuleList([
            nn.Sequential(nn.Conv2d(in_ch, out_ch, 1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))
        ] + [
            nn.Sequential(nn.Conv2d(in_ch, out_ch, 3, padding=r, dilation=r, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))
            for r in rates[1:]
        ])
        self.project = nn.Sequential(
            nn.Conv2d(out_ch * len(rates), out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True), nn.Dropout(0.1)
        )
    def forward(self, x):
        feats = [b(x) for b in self.blocks]
        return self.project(torch.cat(feats, dim=1))

class GatedFusion(nn.Module):
    """ Skip Connection의 노이즈를 필터링하는 게이트 """
    def __init__(self, in_channels):
        super().__init__()
        self.gate = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return x * self.gate(x)

class CrossAttention(nn.Module):
    def __init__(self, dim, num_heads=8, qkv_bias=False):
        super().__init__()
        self.num_heads = num_heads
        self.scale = (dim // num_heads) ** -0.5
        self.q = nn.Linear(dim, dim, bias=qkv_bias)
        self.kv = nn.Linear(dim, dim * 2, bias=qkv_bias)
        self.proj = nn.Linear(dim, dim)

    def forward(self, x, context):
        B, C, H, W = x.shape
        x_flat = x.flatten(2).transpose(1, 2)
        ctx_flat = context.flatten(2).transpose(1, 2)

        Q = self.q(x_flat).reshape(B, -1, self.num_heads, C // self.num_heads).permute(0, 2, 1, 3)

        # ☠  수정완료: Permute 인덱스 중복 해결
        KV = self.kv(ctx_flat).reshape(B, -1, 2, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        K, V = KV[0], KV[1]

        attn = (Q @ K.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)

        x_attn = (attn @ V).transpose(1, 2).reshape(B, -1, C)
        x_attn = self.proj(x_attn).transpose(1, 2).reshape(B, C, H, W)
        return x + x_attn

class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, mlp_ratio=4., dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, int(embed_dim * mlp_ratio)),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(int(embed_dim * mlp_ratio), embed_dim),
            nn.Dropout(dropout)
        )
    def forward(self, x):
        x = x + self.attn(self.norm1(x), self.norm1(x), self.norm1(x))[0]
        x = x + self.mlp(self.norm2(x))
        return x

class ModifiedTransUNet_Final(nn.Module):
    def __init__(self, in_channels=5, num_classes=1, embed_dim=512):
        super().__init__()
        resnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

        # Stem (SE 제거, 5채널 대응)
        self.conv1 = nn.Conv2d(in_channels, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = resnet.bn1
        self.relu = resnet.relu
        self.maxpool = resnet.maxpool

        self.layer1 = resnet.layer1
        self.layer2 = resnet.layer2
        self.layer3 = resnet.layer3
        self.layer4 = resnet.layer4

        # ASPP Module
        self.aspp = ASPP(in_ch=512, out_ch=embed_dim)

        # Transformer Bottleneck
        self.transformer = nn.Sequential(*[
            TransformerBlock(embed_dim, num_heads=8) for _ in range(4)
        ])

        # Gated Fusion
        self.gate3 = GatedFusion(256)
        self.gate2 = GatedFusion(128)
        self.gate1 = GatedFusion(64)

        # Decoder with Cross Attention
        self.up4 = nn.ConvTranspose2d(embed_dim, 256, 2, stride=2)
        self.ca4 = CrossAttention(256)
        self.dec4 = ConvBlock(512, 256)

        self.up3 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.ca3 = CrossAttention(128)
        self.dec3 = ConvBlock(256, 128)

        self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.ca2 = CrossAttention(64)
        self.dec2 = ConvBlock(128, 64)

        self.up1 = nn.ConvTranspose2d(64, 64, 2, stride=2)
        self.dec1 = ConvBlock(128, 64)

        self.final = nn.Sequential(
            nn.ConvTranspose2d(64, 32, 2, stride=2),
            ConvBlock(32, 32),
            nn.Conv2d(32, num_classes, 1)
        )

    def forward(self, x):
        # Stem (128x128 추출)
        x_stem = self.conv1(x)
        x_stem = self.bn1(x_stem)
        x_stem = self.relu(x_stem) # [B, 64, 128, 128]

        # Encoder
        x0 = self.maxpool(x_stem)  # [B, 64, 64, 64]
        x1 = self.layer1(x0)
        x2 = self.layer2(x1)
        x3 = self.layer3(x2)
        x4 = self.layer4(x3) # [B, 512, H/32, W/32]

        # ASPP -> Transformer
        x_aspp = self.aspp(x4)
        B, C, H, W = x_aspp.shape
        x_trans = self.transformer(x_aspp.flatten(2).transpose(1, 2)).transpose(1, 2).reshape(B, C, H, W)

        # Decoder
        d4 = self.up4(x_trans)
        skip3 = self.gate3(x3) # Gated
        d4 = self.dec4(torch.cat([d4, self.ca4(d4, skip3)], dim=1)) # Cross Attn

        d3 = self.up3(d4)
        skip2 = self.gate2(x2)
        d3 = self.dec3(torch.cat([d3, self.ca3(d3, skip2)], dim=1))

        d2 = self.up2(d3)
        skip1 = self.gate1(x1)
        d2 = self.dec2(torch.cat([d2, self.ca2(d2, skip1)], dim=1))

        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, x_stem], dim=1)) # ResNet Stem 결합

        return self.final(d1)

# ============================================================
# 4. 학습 및 실행
# ============================================================
class BCEDiceLoss(nn.Module):
    def __init__(self): super().__init__()
    def forward(self, input, target):
        bce = F.binary_cross_entropy_with_logits(input, target)
        smooth = 1e-5
        input = torch.sigmoid(input).view(-1)
        target = target.view(-1)
        inter = (input * target).sum()
        dice = (2. * inter + smooth) / (input.sum() + target.sum() + smooth)
        return 0.5 * bce + 0.5 * (1 - dice)

def calculate_miou(pred, target, threshold=0.5):
    pred = (torch.sigmoid(pred) > threshold).float()
    intersection = (pred * target).sum()
    union = pred.sum() + target.sum() - intersection
    return (intersection + 1e-6) / (union + 1e-6)

def train_one_epoch(model, loader, optimizer, scaler, device):
    model.train()
    loss_val = 0
    for x, y in tqdm(loader, desc="Train", leave=False):
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            out = model(x)
            loss = BCEDiceLoss()(out, y)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        loss_val += loss.item()
    return loss_val / len(loader)

@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    loss_val, miou_val = 0, 0
    for x, y in tqdm(loader, desc="Valid", leave=False):
        x, y = x.to(device), y.to(device)
        with torch.cuda.amp.autocast():
            out = model(x)
            loss = BCEDiceLoss()(out, y)
        loss_val += loss.item()
        miou_val += calculate_miou(out, y).item()
    return loss_val / len(loader), miou_val / len(loader)

class EarlyStopping:
    def __init__(self, patience=10, min_delta=0.0, mode="max"):
        self.patience = patience
        self.min_delta = min_delta
        self.best = None
        self.num_bad = 0
        self.should_stop = False
        self.mode = mode # mode 속성 추가

    def step(self, value):
        if self.best is None:
            self.best = value; return True
        # 'mode'를 'self.mode'로 수정
        improved = (value > self.best + self.min_delta) if self.mode == "max" else (value < self.best - self.min_delta)
        if improved:
            self.best, self.num_bad = value, 0; return True
        else:
            self.num_bad += 1
            if self.num_bad >= self.patience: self.should_stop = True
            return False

def main():
    set_seed(CFG["SEED"])
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"✅ Device: {device}")

    root = Path(CFG["DATA_ROOT"])

    if CFG["AUTO_COMPUTE_AUX_STATS"]:
        print("⏳ Calculating stats (Log Scale)...")
        no2_mean, no2_std = collect_aux_stats(root / CFG["TRAIN_NO2_DIR"])
        air_mean, air_std = collect_aux_stats(root / CFG["TRAIN_AIR_DIR"])
        CFG["NO2_MEAN"], CFG["NO2_STD"] = no2_mean, no2_std
        CFG["AIR_MEAN"], CFG["AIR_STD"] = air_mean, air_std
        print(f"   NO2: mean={no2_mean:.4f}, std={no2_std:.4f}")
        print(f"   AIR: mean={air_mean:.4f}, std={air_std:.4f}")

    print("🚀 Building Loaders...")
    tr_items = list_pairs_with_aux(root/CFG["TRAIN_IMG_DIR"], root/CFG["TRAIN_MASK_DIR"], root/CFG["TRAIN_NO2_DIR"], root/CFG["TRAIN_AIR_DIR"], CFG["IMAGE_EXT"], CFG["AUX_EXTS"])
    va_items = list_pairs_with_aux(root/CFG["VALID_IMG_DIR"], root/CFG["VALID_MASK_DIR"], root/CFG["VALID_NO2_DIR"], root/CFG["VALID_AIR_DIR"], CFG["IMAGE_EXT"], CFG["AUX_EXTS"])

    train_ds = SN10MultiModal5chDataset(tr_items, img_size=CFG["IMG_SIZE"], is_train=True)
    valid_ds = SN10MultiModal5chDataset(va_items, img_size=CFG["IMG_SIZE"], is_train=False)

    train_loader = DataLoader(train_ds, batch_size=CFG["BATCH_SIZE"], shuffle=True, num_workers=CFG["NUM_WORKERS"], pin_memory=True)
    valid_loader = DataLoader(valid_ds, batch_size=CFG["BATCH_SIZE"], shuffle=False, num_workers=CFG["NUM_WORKERS"], pin_memory=True)

    print("🔥 Model: Modified TransUNet + ASPP + Gated Fusion + Cross Attention (No SEBlock)")
    model = ModifiedTransUNet_Final(in_channels=5).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["LR"], weight_decay=CFG["WD"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG["EPOCHS"], eta_min=1e-6)
    scaler = torch.cuda.amp.GradScaler()

    es = EarlyStopping(patience=CFG["ES_PATIENCE"], min_delta=CFG["ES_MIN_DELTA"], mode="max")
    save_path = Path(CFG["DATA_ROOT"]) / CFG["CHECKPOINT"]

    for epoch in range(1, CFG["EPOCHS"]+1):
        tr_loss = train_one_epoch(model, train_loader, optimizer, scaler, device)
        va_loss, va_miou = evaluate(model, valid_loader, device)
        scheduler.step()

        print(f"[Epoch {epoch:03d}] Train Loss: {tr_loss:.4f} | Valid Loss: {va_loss:.4f} | mIoU: {va_miou:.4f}")

        if es.step(va_miou):
            torch.save(model.state_dict(), save_path)
            print(f"  ★ Best mIoU Saved! ({es.best:.4f})")

        if es.should_stop:
            print("🛑 Early Stopping!")
            break

    print(f"🏆 Final Best mIoU: {es.best:.4f}")

if __name__ == "__main__":
    main()


✅ Device: cuda
⏳ Calculating stats (Log Scale)...


[Stats] TS_SN10_GEMS:   0%|          | 0/8005 [00:00<?, ?it/s]

[Stats] TS_SN10_AIR_Pollution:   0%|          | 0/24000 [00:00<?, ?it/s]

   NO2: mean=0.8381, std=0.1683
   AIR: mean=0.0257, std=0.0815
🚀 Building Loaders...
🔥 Model: Modified TransUNet + ASPP + Gated Fusion + Cross Attention (No SEBlock)


/tmp/ipython-input-3382280816.py:486: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


...

[Epoch 149] Train Loss: 0.0510 | Valid Loss: 0.0666 | mIoU: 0.8851


Train:   0%|          | 0/1000 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x79fbc9551ee0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1654, in __del__
Exception ignored in:     <function _MultiProcessingDataLoaderIter.__del__ at 0x79fbc9551ee0>self._shutdown_workers()

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1654, in __del__
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1637, in _shutdown_workers
        self._shutdown_workers()if w.is_alive():

    File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1637, in _shutdown_workers
     if w.is_alive(): 
     ^ ^ ^  ^^^ ^^^^^^^^^^^
^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^^    ^assert self._parent_pid == os.getpid(), 'can only test a child process'^
 ^ ^ 
  File "/usr/lib/pyt

Valid:   0%|          | 0/125 [00:00<?, ?it/s]

[Epoch 150] Train Loss: 0.0510 | Valid Loss: 0.0668 | mIoU: 0.8852
🏆 Final Best mIoU: 0.8851
